<a href="https://colab.research.google.com/github/waelbakir/CC_RAG/blob/main/CC_RAG1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib plotly seaborn xgboost
!pip install -q langchain langchain-community langchain-huggingface langchain-core
!pip install -q faiss-cpu sentence-transformers transformers accelerate

print("All libraries installed successfully!")

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# ==========================================
# CELL 2: GENERATE REALISTIC TEXT DATA FOR RAG
# ==========================================
import os

project_path = '/content/drive/MyDrive/Fraud_RAG_Project'
if not os.path.exists(project_path):
    os.makedirs(project_path)

guidelines_text = """
Bank Standard Operating Procedures for Fraud Detection:

Policy 101 - Velocity Rule (BIN Attack): Small transactions under $5.00 following a long period of inactivity indicate a Bank Identification Number (BIN) attack, where botnets test if stolen cards are active. Action: Decline the transaction, trigger 3D Secure (3DS) authentication, and hold the account until SMS validation.

Policy 102 - Card-Not-Present (CNP) High-Risk: If an online CNP transaction exceeds $500, occurs during late night hours, and the Machine Learning Risk Score is 'highest' (over 80%), this indicates an Account Takeover. Action: Block payment immediately to prevent chargeback liability shift to the bank, and escalate to Level 2 fraud analysts.

Policy 103 - Cross-Border Mismatch: If the shipping or IP address does not match the Card Issuing Country, and the anomaly probability is elevated, it indicates offshore fraud rings. Action: Request Adaptive 3D Secure (3DS). If authentication fails, reject the authorization.

Policy 104 - Zero-Dollar Authorization: Transactions for exactly $0.00 with high anomaly scores indicate a fraudster attempting to link the credit card to a new digital wallet (like Apple Pay or Google Pay) without the physical card. Action: Reject the authorization and force a complete account password reset.
"""

file_path = f"{project_path}/bank_fraud_guidelines.txt"
with open(file_path, "w") as file:
    file.write(guidelines_text)

print("Authentic Bank Guidelines successfully saved to Google Drive!")

Authentic Bank Guidelines successfully saved to Google Drive!


In [ ]:
# ==========================================
# CELL 3: DATA PREP & 3D PCA VISUALIZATION
# ==========================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px

print("Loading Data from Google Drive...")

# 1. Load Data from Drive
df = pd.read_csv('/content/drive/MyDrive/Fraud_RAG_Project/creditcard.csv')

# Clean any accidentally corrupted rows
df = df.dropna()
print(f"Data loaded successfully! Total valid rows: {len(df)}")

X = df.drop(columns=['Class'])
y = df['Class']

# 2. Scale the data (Required for PCA and Machine Learning)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. PCA for Visualization Only
print("Compressing 30 dimensions to 3 using PCA...")
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

# 4. Prepare a sample for the 3D Graph (Plotting all dots crashes the browser)
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2', 'PC3'])
pca_df['Class'] = y.values

# Grab all frauds, but only 5000 normal transactions for the plot
plot_df = pd.concat([
    pca_df[pca_df['Class'] == 0].sample(n=5000, random_state=42),
    pca_df[pca_df['Class'] == 1]
])
plot_df['Label'] = plot_df['Class'].map({0: 'Normal', 1: 'Fraud'})

# 5. Render 3D Plot
fig = px.scatter_3d(plot_df, x='PC1', y='PC2', z='PC3', color='Label',
                    color_discrete_map={'Normal': 'blue', 'Fraud': 'red'},
                    title="PCA: 3D Visualization of Credit Card Fraud", opacity=0.7)
fig.show()

In [ ]:
# ==========================================
# CELL 4: XGBOOST ALGORITHM TRAINING
# ==========================================
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("Initializing XGBoost with Cost-Sensitive Learning...")

# 1. Split Data (Using the fully scaled 30-dimensional data, NOT the 3D PCA data!)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# 2. The Math of Imbalance: Calculate the Cost Penalty
penalty_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"XGBoost is mathematically penalized {penalty_weight:.2f}x more for missing fraud!")

# 3. Train the Model
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=penalty_weight, # This replaces the need for SMOTE!
    max_depth=5,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

# 4. Predict and Evaluate
predictions = xgb_model.predict(X_test)
print("\n--- XGBoost Evaluation ---")
print(classification_report(y_test, predictions, target_names=['Normal', 'Fraud']))

# 5. Plot Confusion Matrix
cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Predicted Normal', 'Predicted Fraud'],
            yticklabels=['Actual Normal', 'Actual Fraud'])
plt.title('XGBoost Confusion Matrix')
plt.show()

In [ ]:
# Install all required AI and LangChain libraries for this new notebook
!pip install -q langchain langchain-community langchain-huggingface langchain-core
!pip install -q faiss-cpu sentence-transformers transformers accelerate

In [10]:
!pip install -q langchain-text-splitters

In [ ]:
# ==========================================
# CELL 5: BUILD VECTOR DB & DETERMINISTIC AI
# ==========================================
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import pipeline
import torch

print("Loading AI Models and Chunking Data...")

# 1. Load Data
loader = TextLoader('/content/drive/MyDrive/Fraud_RAG_Project/bank_fraud_guidelines.txt')
documents = loader.load()

# 2. CHUNKING ALGORITHM
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0, separators=["Policy"])
docs = text_splitter.split_documents(documents)

# 3. Build Vector DB
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(docs, embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 1})

# 4. Load TinyLlama with GREEDY DECODING (Zero Creativity!)
llm_pipeline = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    max_new_tokens=150,
    return_full_text=False,
    do_sample=False,  # <--- THIS KILLS HALLUCINATIONS!
    device=0
)
llm = HuggingFacePipeline(pipeline=llm_pipeline)

# 5. The Bulletproof Prompt
prompt_template = """<|system|>
You are a strict banking algorithm. Read the Bank Policy below. Answer the user using ONLY the information in the policy. Do not invent any rules.
Bank Policy: {context}</s>
<|user|>
{question}</s>
<|assistant|>
Based on the policy provided, the fraud type is"""

PROMPT = PromptTemplate.from_template(prompt_template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

print("Deterministic RAG Pipeline Built Successfully!")

In [20]:
import numpy as np

# Find all actual fraud indices in the test set
fraud_indices = np.where(y_test == 1)[0]

print("=== FRAUD TRANSACTION DIRECTORY ===")
for idx in fraud_indices:
    original_idx = y_test.index[idx]
    amt = X.loc[original_idx, 'Amount']

    # We will only print out a few distinct ones so you can pick!
    if amt == 0.00:
        print(f"Index {idx} -> Zero Dollar Fraud ($0.00)")
    elif 0.01 <= amt <= 5.00:
        print(f"Index {idx} -> Micro-Auth Fraud (${amt:.2f})")
    elif amt > 500.00:
        print(f"Index {idx} -> High-Dollar Fraud (${amt:.2f})")

=== FRAUD TRANSACTION DIRECTORY ===
Index 840 -> Micro-Auth Fraud ($0.01)
Index 1146 -> Micro-Auth Fraud ($1.00)
Index 5453 -> Micro-Auth Fraud ($1.00)
Index 7299 -> Micro-Auth Fraud ($1.63)
Index 7337 -> Micro-Auth Fraud ($1.00)
Index 9036 -> High-Dollar Fraud ($512.25)
Index 9730 -> Micro-Auth Fraud ($1.00)
Index 10130 -> Micro-Auth Fraud ($1.18)
Index 12588 -> Micro-Auth Fraud ($1.00)
Index 16303 -> Micro-Auth Fraud ($1.00)
Index 17046 -> Micro-Auth Fraud ($1.00)
Index 19638 -> Micro-Auth Fraud ($1.00)
Index 20687 -> Micro-Auth Fraud ($1.00)
Index 20971 -> Micro-Auth Fraud ($0.76)
Index 20992 -> Micro-Auth Fraud ($1.00)
Index 23090 -> Micro-Auth Fraud ($1.00)
Index 24570 -> High-Dollar Fraud ($529.00)
Index 25468 -> Micro-Auth Fraud ($1.00)
Index 26685 -> Micro-Auth Fraud ($0.76)
Index 26892 -> Micro-Auth Fraud ($2.00)
Index 28390 -> High-Dollar Fraud ($519.90)
Index 28867 -> Micro-Auth Fraud ($1.00)
Index 29865 -> Micro-Auth Fraud ($0.77)
Index 30724 -> Micro-Auth Fraud ($1.00)
Ind

In [30]:
# ==========================================
# CELL 6: THE FINAL AI COPILOT FUNCTION (DEBUG MODE)
# ==========================================
import numpy as np

def evaluate_transaction(dataset_index):
    # 1. Fetch Real Data
    real_label = y_test.iloc[dataset_index]
    original_index = y_test.index[dataset_index]
    actual_amount = X.loc[original_index, 'Amount']

    print(f"\n==================================================")
    print(f"🔍 INVESTIGATING TRANSACTION ID: {original_index}")
    print(f"==================================================")
    print(f"💰 Amount: ${actual_amount:.2f}")
    print(f"✅ True Label in Dataset: {'FRAUD' if real_label == 1 else 'NORMAL'}")

    # 2. XGBoost Prediction
    transaction_features = X_test[dataset_index].reshape(1, -1)
    fraud_prob = xgb_model.predict_proba(transaction_features)[0][1] * 100

    print(f"🤖 XGBoost Predicted Fraud Probability: {fraud_prob:.2f}%")

    if fraud_prob < 50.0:
        print("\n✅ DECISION: Transaction Approved. No AI Explanation needed.")
        print("==================================================")
        return

    print("\n🚨 FRAUD DETECTED! Triggering Vector Search and RAG AI...")

    # 3. ADVANCED QUERY AUGMENTATION
    # We use the EXACT words from the text file so FAISS finds a perfect math match.
    if actual_amount == 0.00:
        semantic_keywords = "Policy 104 Zero-Dollar Authorization Apple Pay Account password reset"
    elif actual_amount < 5.00:
        semantic_keywords = "Policy 101 Velocity Rule BIN Attack botnet under $5.00"
    elif actual_amount > 500.00:
        semantic_keywords = "Policy 102 Card-Not-Present CNP High-Risk exceeds $500 Account Takeover"
    else:
        semantic_keywords = "Policy 103 Cross-Border Mismatch offshore fraud Adaptive 3D Secure"

    search_query = f"{semantic_keywords} amount {actual_amount}"

    # --- THE DEBUGGER: What did FAISS actually find? ---
    retrieved_docs = retriever.invoke(search_query)
    print(f"\n[FAISS DEBUG] Retrieved Policy: {retrieved_docs[0].page_content.strip()[:100]}...") # Prints the first 100 characters

    # 4. Generate AI Report
    ai_prompt = f"The transaction amount is ${actual_amount:.2f}. Based strictly on the policy provided, what type of fraud is this and what is the action?"
    report = rag_chain.invoke(ai_prompt)

    print("\n=== AI COPILOT REPORT ===")
    print(report.split("<|assistant|>")[-1].strip())
    print("==================================================")

# ==========================================
# Let's test the $512.25 transaction again!
# ==========================================
# Find it in the test set (Make sure to put the correct test_index here!)
fraud_indices = np.where(y_test == 1)[0]

# Assuming 141260 was the dataset ID, let's just find the first fraud over $500:
high_dollar_frauds =[idx for idx in fraud_indices if X_test[idx][0] > 500] # Adjusting based on how you index

# Let's run a test on the $512.25 transaction or any high-dollar one
# (Just swap the number below with whatever index gave you the $512.25)
evaluate_transaction(fraud_indices[6])  # Replace with your specific index

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 INVESTIGATING TRANSACTION ID: 204064
💰 Amount: $345.00
✅ True Label in Dataset: FRAUD
🤖 XGBoost Predicted Fraud Probability: 99.97%

🚨 FRAUD DETECTED! Triggering Vector Search and RAG AI...

[FAISS DEBUG] Retrieved Policy: Policy 103 - Cross-Border Mismatch: If the shipping or IP address does not match the Card Issuing Co...

=== AI COPILOT REPORT ===
a high anomaly score transaction for exactly $345.00 with a high anomaly score. The action taken is a complete account password reset.
